In [1]:
from IPython.display import display
import time
import hmac
import hashlib
import urllib.request
from urllib.parse import quote
import json
import pandas as pd

In [2]:
API_KEY = "9cc2976dc30ff6190b7b0485536cc483"
SECRET  = "61f2cb7a045b539c7a530af70a5be50cbff571f69e6dd8fbaa3dcc1fb8835ac8"

METHOD = "GET"
PATH   = "prediction/playerSeason"
QUERY  = {"p_no": "12568"}

def normalize_query(params: dict) -> str:
    safe = "-_.!~*'()"
    return "&".join(
        f"{quote(str(k), safe=safe)}={quote(str(params[k]), safe=safe)}"
        for k in sorted(params.keys())
    )

def make_signature(secret: str, payload: str) -> str:
    return hmac.new(
        secret.encode("utf-8"),
        payload.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

normalized = normalize_query(QUERY)
timestamp = str(int(time.time()))

payload = f"{METHOD}|{PATH}|{normalized}|{timestamp}"
signature = make_signature(SECRET, payload)

url = f"https://api.statiz.co.kr/baseballApi/{PATH}?{normalized}"

headers = {
    "X-API-KEY": API_KEY,
    "X-TIMESTAMP": timestamp,
    "X-SIGNATURE": signature,
}

req = urllib.request.Request(url, method=METHOD, headers=headers)

with urllib.request.urlopen(req, timeout=30) as resp:
    body = resp.read().decode("utf-8")

# JSON 변환
json_data = json.loads(body)

# 확인
display(json_data)



{'basic': {'list': [{'p_no': 12568,
    'leagueType': 10100,
    't_code': '6002',
    'p_name': '이영하',
    'year': '2023',
    'p_nameEN': 'Young-Ha Lee',
    'G': '36',
    'GS': '0',
    'GR': '36',
    'GF': '5',
    'CG': '0',
    'SHO': '0',
    'W': '5',
    'L': '3',
    'S': '0',
    'HD': '4',
    'IP': 39.1,
    'ER': '24',
    'R': '27',
    'TBF': '184',
    'H': '40',
    '2B': '9',
    '3B': '1',
    'HR': '2',
    'BB': '21',
    'HP': '4',
    'IB': '1',
    'SO': '28',
    'ROE': '4',
    'BK': '0',
    'WP': '2',
    'ERA': 5.49,
    'FIP': 4.56623,
    'WHIP': 1.55,
    'WAR': -0.3460358763518433,
    'QS': '0',
    'p_position': 1},
   {'p_no': 12568,
    'leagueType': 10100,
    't_code': '6002',
    'p_name': '이영하',
    'year': '2024',
    'p_nameEN': 'Young-Ha Lee',
    'G': '59',
    'GS': '1',
    'GR': '58',
    'GF': '8',
    'CG': '0',
    'SHO': '0',
    'W': '5',
    'L': '4',
    'S': '2',
    'HD': '5',
    'IP': 65.1,
    'ER': '29',
    'R': '34',
   

In [3]:
# json 저장
with open("이영하_12568.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)


In [5]:
# json 불러오기
with open("이영하_12568.json", "r", encoding="utf-8") as f:
    json_data = json.load(f) # 데이터명 'json_data'

In [3]:
json_data.keys()

dict_keys(['basic', 'deepen', 'result_cd', 'result_msg', 'update_time'])

In [4]:
# DataFrame 변환
basic_df = pd.DataFrame(json_data["basic"]["list"])
deepen_df = pd.DataFrame(json_data["deepen"]["list"])

display(basic_df.head())
display(deepen_df.head())

print(basic_df.columns)

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,GS,GR,GF,...,SO,ROE,BK,WP,ERA,FIP,WHIP,WAR,QS,p_position
0,12568,10100,6002,이영하,2023,Young-Ha Lee,36,0,36,5,...,28,4,0,2,5.49,4.56623,1.55,-0.346036,0,1
1,12568,10100,6002,이영하,2024,Young-Ha Lee,59,1,58,8,...,59,6,0,7,3.99,4.56377,1.50,0.447534,0,1
2,12568,10100,6002,이영하,2025,Young-Ha Lee,73,0,73,8,...,72,5,0,7,4.05,4.09264,1.53,0.896688,0,1


,p_no,t_code,p_name,year,K9,BB9,KBB,HR9,OBP,SLG,OPS
0,12568,6002,이영하,2023,6.407,4.805,1.3333,0.458,0.359,0.365,0.724
1,12568,6002,이영하,2024,8.128,4.959,1.6389,0.551,0.339,0.349,0.688
2,12568,6002,이영하,2025,9.720,5.265,1.8462,0.540,0.367,0.343,0.710


Index(['p_no', 'leagueType', 't_code', 'p_name', 'year', 'p_nameEN', 'G', 'GS',
       'GR', 'GF', 'CG', 'SHO', 'W', 'L', 'S', 'HD', 'IP', 'ER', 'R', 'TBF',
       'H', '2B', '3B', 'HR', 'BB', 'HP', 'IB', 'SO', 'ROE', 'BK', 'WP', 'ERA',
       'FIP', 'WHIP', 'WAR', 'QS', 'p_position'],
      dtype='object')


In [9]:
deepen_df["year"] = basic_df["year"].values

merged_df = basic_df.merge(
    deepen_df[["year", "K9", "BB9", "KBB", "HR9", "OBP", "SLG", "OPS"]],
    on="year",
    how="left"
)

display(merged_df.head())
print(merged_df.columns)

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,GS,GR,GF,...,WAR,QS,p_position,K9,BB9,KBB,HR9,OBP,SLG,OPS
0,12568,10100,6002,이영하,2023,Young-Ha Lee,36,0,36,5,...,-0.346036,0,1,6.407,4.805,1.3333,0.458,0.359,0.365,0.724
1,12568,10100,6002,이영하,2024,Young-Ha Lee,59,1,58,8,...,0.447534,0,1,8.128,4.959,1.6389,0.551,0.339,0.349,0.688
2,12568,10100,6002,이영하,2025,Young-Ha Lee,73,0,73,8,...,0.896688,0,1,9.720,5.265,1.8462,0.540,0.367,0.343,0.710


Index(['p_no', 'leagueType', 't_code', 'p_name', 'year', 'p_nameEN', 'G', 'GS',
       'GR', 'GF', 'CG', 'SHO', 'W', 'L', 'S', 'HD', 'IP', 'ER', 'R', 'TBF',
       'H', '2B', '3B', 'HR', 'BB', 'HP', 'IB', 'SO', 'ROE', 'BK', 'WP', 'ERA',
       'FIP', 'WHIP', 'WAR', 'QS', 'p_position', 'K9', 'BB9', 'KBB', 'HR9',
       'OBP', 'SLG', 'OPS'],
      dtype='object')


In [10]:
merged_df.dtypes

p_no            int64
leagueType      int64
t_code         object
p_name         object
year           object
p_nameEN       object
G              object
GS             object
GR             object
GF             object
CG             object
SHO            object
W              object
L              object
S              object
HD             object
IP            float64
ER             object
R              object
TBF            object
H              object
2B             object
3B             object
HR             object
BB             object
HP             object
IB             object
SO             object
ROE            object
BK             object
WP             object
ERA           float64
FIP           float64
WHIP          float64
WAR           float64
QS             object
p_position      int64
K9            float64
BB9           float64
KBB            object
HR9           float64
OBP           float64
SLG           float64
OPS           float64
dtype: object

In [11]:
str_cols = ["p_name","p_nameEN","t_code"]
int_cols = [
    "p_no", "leagueType", "year", "p_position",
    "G", "GS", "GR", "GF", "CG", "SHO", "W", "L", "S", "HD", # 게임수, 선발, 구원, 마무리, 완투, 완봉, 승무세홀
    "ER", "R", "TBF", # 자책, 실점, 상대타자수
    "H", "2B", "3B", "HR",
    "BB", "HP", "IB", "SO", # IB(고의사구)
    "ROE", "BK", "WP", # 실책출루, 보크, 폭투
    "QS"
]
float_cols = [
    "IP", "ERA", "FIP", "WHIP", "WAR", # IP(이닝), FIP(수비무관평자)
    "K9", "BB9", "KBB", "HR9",
    "OBP", "SLG", "OPS" # 피출루율, 장타율, 옵스
]

merged_df[str_cols] = merged_df[str_cols].astype("object")

for c in int_cols:
    if c in merged_df.columns:
        merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce").astype("int64")

for c in float_cols:
    if c in merged_df.columns:
        merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce")

In [12]:
display(merged_df)

merged_df.dtypes


,p_no,leagueType,t_code,p_name,year,p_nameEN,G,GS,GR,GF,...,WAR,QS,p_position,K9,BB9,KBB,HR9,OBP,SLG,OPS
0,12568,10100,6002,이영하,2023,Young-Ha Lee,36,0,36,5,...,-0.346036,0,1,6.407,4.805,1.3333,0.458,0.359,0.365,0.724
1,12568,10100,6002,이영하,2024,Young-Ha Lee,59,1,58,8,...,0.447534,0,1,8.128,4.959,1.6389,0.551,0.339,0.349,0.688
2,12568,10100,6002,이영하,2025,Young-Ha Lee,73,0,73,8,...,0.896688,0,1,9.720,5.265,1.8462,0.540,0.367,0.343,0.710


p_no            int64
leagueType      int64
t_code         object
p_name         object
year            int64
p_nameEN       object
G               int64
GS              int64
GR              int64
GF              int64
CG              int64
SHO             int64
W               int64
L               int64
S               int64
HD              int64
IP            float64
ER              int64
R               int64
TBF             int64
H               int64
2B              int64
3B              int64
HR              int64
BB              int64
HP              int64
IB              int64
SO              int64
ROE             int64
BK              int64
WP              int64
ERA           float64
FIP           float64
WHIP          float64
WAR           float64
QS              int64
p_position      int64
K9            float64
BB9           float64
KBB           float64
HR9           float64
OBP           float64
SLG           float64
OPS           float64
dtype: object

In [ ]:
merged_df.to_csv(
    "playerSeason_12568.csv",
    index=False,
    encoding="utf-8-sig"
)

In [13]:
print(merged_df[["WHIP", "BB"]])

   WHIP  BB
0  1.55  21
1  1.50  36
2  1.53  39
